# RISK ANALYTICS — 05 Controles de Calidad y Trazabilidad

**Alcance:** validación del pipeline completo Bronze → Silver → Gold sobre los datos reales.
**Resultado:** tabla de resultados consultable `eventosriesgoyvar.gold.controles_calidad` + evidencia antes/después del control de re-ejecución en `controles_snapshot`.

Controles solicitados:
1. Identificación de registros sin ID o con identificadores inválidos
2. Consistencia de fechas y montos
3. Registros duplicados
4. Campos obligatorios incompletos
5. Consistencia de las clasificaciones
6. Conciliación de cantidades de registros Bronze-Silver-Gold (con las transformaciones justificadas)
7. Consistencia de los montos agregados
8. Comportamiento del pipeline cuando se ejecuta nuevamente (idempotencia)

**Reglas de negocio:** NO se modifica ninguna regla aprobada. Los controles solo leen; las excepciones se reportan con su explicación.
Categorías de resultado: **OK** (cumple), **EXCEPCION** (desviación real pero documentada/justificada por una regla aprobada), **FALLA** (desviación no justificada → requiere atención).

El control 8 re-ejecuta los notebooks 03 (Silver) y 04 (Gold) SIN cambiar reglas y compara métricas de negocio antes/después. Las columnas de lote (_lote_silver, _lote_gold) cambian por diseño en cada ejecución: son la evidencia de trazabilidad del re-procesamiento.

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

CATALOGO = "eventosriesgoyvar"
TABLA_BRONZE = f"{CATALOGO}.bronze.eventos_riesgo_raw"
TABLA_SILVER = f"{CATALOGO}.silver.eventos_riesgo"
SCHEMA_GOLD = f"{CATALOGO}.gold"
LOTE_QA = f"qa_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

bronze = spark.table(TABLA_BRONZE)
silver = spark.table(TABLA_SILVER)
g = lambda t: spark.table(f"{SCHEMA_GOLD}.{t}")
FILAS_SILVER = silver.count()

# Framework de controles: acumula los resultados y los imprime en línea
RESULTADOS = []

def control(id_control, capa, categoria, descripcion, resultado, detalle=""):
    RESULTADOS.append({"id_control": id_control, "capa": capa, "categoria": categoria,
                       "descripcion": descripcion, "resultado": resultado, "detalle": str(detalle),
                       "fecha_ejecucion": datetime.now().date(), "_lote_qa": LOTE_QA})
    print(f"{resultado:9s} {id_control:4s} {descripcion}" + (f" | {detalle}" if detalle else ""))

print("Controles de calidad sobre:", TABLA_BRONZE, "->", TABLA_SILVER, "->", SCHEMA_GOLD + ".<tablas>")
print("Filas Silver:", FILAS_SILVER, "| Lote QA:", LOTE_QA)

Controles de calidad sobre: eventosriesgoyvar.bronze.eventos_riesgo_raw -> eventosriesgoyvar.silver.eventos_riesgo -> eventosriesgoyvar.gold.<tablas>
Filas Silver: 51889 | Lote QA: qa_20261121_210449


In [0]:
# ===== CONTROL 1: IDENTIFICADORES (SIN ID O INVÁLIDOS) =====
# 1a. id_evento (clave técnica generada en Silver): sin nulos, sin duplicados, continua 1..51.889
r = silver.agg(F.count("*").alias("n"), F.countDistinct("id_evento").alias("d"),
               F.min("id_evento").alias("mn"), F.max("id_evento").alias("mx"),
               F.sum(F.when(F.col("id_evento").isNull(), 1).otherwise(0)).alias("nulos")).collect()[0]
control("1a", "SILVER", "Identificadores",
        "id_evento: sin nulos, sin duplicados, continuo 1..51.889",
        "OK" if (r["n"] == r["d"] == r["mx"] == 51889 and r["mn"] == 1 and r["nulos"] == 0) else "FALLA",
        f"filas={r['n']}, distintos={r['d']}, nulos={r['nulos']}, rango={r['mn']}..{r['mx']}")

# 1b. idinsumo (único identificador de negocio de la fuente): nulos y repetidos
r = silver.agg(F.sum(F.when(F.col("idinsumo").isNull(), 1).otherwise(0)).alias("nulos"),
               F.countDistinct("idinsumo").alias("distintos")).collect()[0]
presentes = FILAS_SILVER - r["nulos"]
repetidos = presentes - r["distintos"]
control("1b", "SILVER", "Identificadores",
        "idinsumo: nulos y valores repetidos (limitación de la fuente, documentada en Silver)",
        "EXCEPCION",
        f"nulos={r['nulos']} (44,4%), presentes={presentes}, filas_repetidas={repetidos} (exceso sobre valores distintos: idinsumo NO es clave única, por eso Silver genera id_evento)")

# 1c. Gold modelo_insumo_severidad: id_evento válido (existe en Silver, sin duplicados)
ms = g("modelo_insumo_severidad")
r = ms.agg(F.count("*").alias("n"), F.countDistinct("id_evento").alias("d"),
           F.sum(F.when(F.col("id_evento").isNull(), 1).otherwise(0)).alias("nulos")).collect()[0]
huerfanos = ms.join(silver.select("id_evento"), "id_evento", "left_anti").count()
control("1c", "GOLD", "Identificadores",
        "modelo_insumo_severidad: id_evento presente, único y con registro padre en Silver",
        "OK" if (r["n"] == r["d"] and r["nulos"] == 0 and huerfanos == 0) else "FALLA",
        f"filas={r['n']}, distintos={r['d']}, nulos={r['nulos']}, sin_padre_en_silver={huerfanos}")

# ===== CONTROL 2: CONSISTENCIA DE FECHAS Y MONTOS =====
# 2a. fecha_inicio presente en el 100% de las filas (fecha obligatoria del evento)
nulos_fi = silver.filter(F.col("fecha_inicio").isNull()).count()
control("2a", "SILVER", "Fechas", "fecha_inicio presente en todas las filas",
        "OK" if nulos_fi == 0 else "FALLA", f"nulos={nulos_fi}")

# 2b. coherencia temporal inicio/fin (donde ambas fechas existen)
ambas = silver.filter(F.col("fecha_fin").isNotNull()).count()
viol_orden = silver.filter(F.col("fecha_fin") < F.col("fecha_inicio")).count()
control("2b", "SILVER", "Fechas", "fecha_fin >= fecha_inicio cuando ambas existen",
        "EXCEPCION" if viol_orden > 0 else "OK",
        f"violaciones={viol_orden} de {ambas} filas con ambas fechas (se conservan tal cual: regla aprobada de no corrección de valores)")

# 2c. fechas dentro del rango esperado del histórico (2000-2026)
fuera = silver.filter((F.col("fecha_inicio") < F.lit("2000-01-01").cast("date")) |
                      (F.col("fecha_inicio") > F.lit("2026-12-31").cast("date"))).count()
control("2c", "SILVER", "Fechas", "fecha_inicio dentro del rango 2000-2026",
        "OK" if fuera == 0 else "EXCEPCION", f"fuera_de_rango={fuera}")

# 2d. montos: sin negativos en bruta/recuperadas ni ceros; netas negativas documentadas y marcadas
neg = silver.agg(F.sum(F.when(F.col("cuantia_bruta") < 0, 1).otherwise(0)).alias("bruta"),
                  F.sum(F.when(F.col("cuantia_total_recuperada") < 0, 1).otherwise(0)).alias("rec"),
                  F.sum(F.when(F.col("cuantia_neta") < 0, 1).otherwise(0)).alias("neta"),
                  F.sum(F.when(F.col("cuantia_bruta") == 0, 1).otherwise(0)).alias("ceros")).collect()[0]
ok2d = neg["bruta"] == 0 and neg["rec"] == 0 and neg["ceros"] == 0 and neg["neta"] == 28
control("2d", "SILVER", "Montos",
        "Sin negativos en bruta/recuperada ni ceros; 28 netas negativas documentadas (flg_neta_negativa)",
        "OK" if ok2d else "FALLA",
        f"bruta_neg={neg['bruta']}, recuperada_neg={neg['rec']}, ceros_bruta={neg['ceros']}, neta_neg={neg['neta']}")

# 2e. consistencia aritmética neta = bruta - recuperada (donde las tres cuantías existen)
con_tres = silver.filter(F.col("cuantia_bruta").isNotNull() & F.col("cuantia_neta").isNotNull() &
                         F.col("cuantia_total_recuperada").isNotNull())
n_tres = con_tres.count()
inc_arit = con_tres.filter(F.col("cuantia_neta") != F.col("cuantia_bruta") - F.col("cuantia_total_recuperada")).count()
control("2e", "SILVER", "Montos", "neta == bruta - recuperada donde las tres cuantías existen",
        "EXCEPCION" if inc_arit > 0 else "OK",
        f"inconsistencias={inc_arit} de {n_tres} filas (flg_inconsistencia_neta; regla: los valores no se corrigen)")

# 2f. moneda única PESOS
otras_mon = silver.filter(F.col("moneda") != "PESOS").count()
control("2f", "SILVER", "Montos", "moneda == PESOS en todas las filas",
        "OK" if otras_mon == 0 else "FALLA", f"otras_monedas={otras_mon}")

OK        1a   id_evento: sin nulos, sin duplicados, continuo 1..51.889 | filas=51889, distintos=51889, nulos=0, rango=1..51889
EXCEPCION 1b   idinsumo: nulos y valores repetidos (limitación de la fuente, documentada en Silver) | nulos=23026 (44,4%), presentes=28863, filas_repetidas=513 (exceso sobre valores distintos: idinsumo NO es clave única, por eso Silver genera id_evento)
OK        1c   modelo_insumo_severidad: id_evento presente, único y con registro padre en Silver | filas=2046, distintos=2046, nulos=0, sin_padre_en_silver=0
OK        2a   fecha_inicio presente en todas las filas | nulos=0
OK        2b   fecha_fin >= fecha_inicio cuando ambas existen | violaciones=0 de 51889 filas con ambas fechas (se conservan tal cual: regla aprobada de no corrección de valores)
OK        2c   fecha_inicio dentro del rango 2000-2026 | fuera_de_rango=0
OK        2d   Sin negativos en bruta/recuperada ni ceros; 28 netas negativas documentadas (flg_neta_negativa) | bruta_neg=0, recuperada_neg=0

In [0]:
# ===== CONTROL 3: REGISTROS DUPLICADOS =====
# Regla aprobada (20-Nov-2026): NO deduplicar; filas con valores idénticos son eventos distintos.
# Se reportan como EXCEPCION documentada; nunca se eliminan.

# 3a. clave técnica id_evento sin duplicados
dup_id = silver.groupBy("id_evento").count().filter(F.col("count") > 1).count()
control("3a", "SILVER", "Duplicados", "id_evento sin duplicados (clave técnica)",
        "OK" if dup_id == 0 else "FALLA", f"ids_duplicados={dup_id}")

# 3b. filas idénticas en columnas de negocio (sin id_evento ni trazabilidad): se conservan por regla
cols_negocio = [c for c in silver.columns if not c.startswith("_") and c != "id_evento"]
r = (silver.groupBy(*cols_negocio).count().filter(F.col("count") > 1)
     .agg(F.count("*").alias("grupos"), F.sum("count").alias("filas")).collect()[0])
grupos_dup = r["grupos"] or 0
filas_dup = r["filas"] or 0
control("3b", "SILVER", "Duplicados",
        "Filas idénticas en columnas de negocio: se conservan (regla aprobada: son eventos distintos)",
        "EXCEPCION" if grupos_dup > 0 else "OK",
        f"grupos_repetidos={grupos_dup}, filas_involucradas={filas_dup} (se conservan todas; ninguna eliminada)")

# 3c. Gold: claves de agregación sin duplicados
claves_gold = [("freq_eventos_periodo", ["periodo"]), ("perdidas_por_periodo", ["periodo"]),
               ("series_frecuencia_severidad", ["periodo"]), ("freq_por_proceso", ["codigo_proceso", "proceso"]),
               ("freq_por_categoria", ["cod_categoria_n1"]), ("freq_por_causa", ["cod_causa"]),
               ("freq_por_severidad", ["nivel_severidad"]), ("perdidas_por_segmento", ["anio_evento", "cod_categoria_n1"]),
               ("modelo_insumo_frecuencia", ["cod_categoria_n1", "periodo"])]
det3c = []
ok3c = True
for tabla, cols in claves_gold:
    d = g(tabla).groupBy(*cols).count().filter(F.col("count") > 1).count()
    ok3c &= (d == 0)
    det3c.append(f"{tabla}={d}")
control("3c", "GOLD", "Duplicados", "Claves de agregación de las tablas Gold sin duplicados",
        "OK" if ok3c else "FALLA", "; ".join(det3c))

# ===== CONTROL 4: CAMPOS OBLIGATORIOS INCOMPLETOS =====
# 4a. campos obligatorios del evento (regla de negocio): 100% presentes en Silver
OBLIGATORIOS = ["fecha_inicio", "tipo_evento", "codigo_perdida", "cod_categoria_n1",
                "categoria_n1", "anio_evento", "mes_evento"]
faltan = {c: silver.filter(F.col(c).isNull()).count() for c in OBLIGATORIOS}
control("4a", "SILVER", "Obligatorios", "Campos obligatorios del evento 100% presentes",
        "OK" if all(v == 0 for v in faltan.values()) else "FALLA",
        "; ".join(f"{c}={v}" for c, v in faltan.items()))

# 4b. completitud de campos analíticos (informativo: ausencias estructurales, nunca imputadas)
CAMPOS_INFO = ["cuantia_bruta", "cuantia_neta", "fecha_contabilizacion", "fecha_recuperacion", "idinsumo", "canal", "ciudad"]
det4b = []
for c in CAMPOS_INFO:
    n = silver.filter(F.col(c).isNull()).count()
    det4b.append(f"{c}={n/51889*100:.1f}% nulos")
control("4b", "SILVER", "Obligatorios",
        "Completitud de campos analíticos (ausencias estructurales de la fuente, no imputadas)",
        "INFO", "; ".join(det4b))

OK        3a   id_evento sin duplicados (clave técnica) | ids_duplicados=0
EXCEPCION 3b   Filas idénticas en columnas de negocio: se conservan (regla aprobada: son eventos distintos) | grupos_repetidos=953, filas_involucradas=1985 (se conservan todas; ninguna eliminada)
OK        3c   Claves de agregación de las tablas Gold sin duplicados | freq_eventos_periodo=0; perdidas_por_periodo=0; series_frecuencia_severidad=0; freq_por_proceso=0; freq_por_categoria=0; freq_por_causa=0; freq_por_severidad=0; perdidas_por_segmento=0; modelo_insumo_frecuencia=0
OK        4a   Campos obligatorios del evento 100% presentes | fecha_inicio=0; tipo_evento=0; codigo_perdida=0; cod_categoria_n1=0; categoria_n1=0; anio_evento=0; mes_evento=0
INFO      4b   Completitud de campos analíticos (ausencias estructurales de la fuente, no imputadas) | cuantia_bruta=48.5% nulos; cuantia_neta=48.8% nulos; fecha_contabilizacion=96.0% nulos; fecha_recuperacion=99.5% nulos; idinsumo=44.4% nulos; canal=0.0% nulos; ciuda

In [0]:
# ===== CONTROL 5: CONSISTENCIA DE CLASIFICACIONES =====
# 5a. tipo_evento <-> codigo_perdida perfectamente alineados (P-B, R-A)
viol5a = silver.filter(~(((F.col("tipo_evento") == "P") & (F.col("codigo_perdida") == "B")) |
                          ((F.col("tipo_evento") == "R") & (F.col("codigo_perdida") == "A")))).count()
control("5a", "SILVER", "Clasificaciones", "tipo_evento P/B y R/A perfectamente alineados",
        "OK" if viol5a == 0 else "FALLA", f"violaciones={viol5a}")

# 5b. dominios cerrados: canal=7, severidad=4 (más NULL), categorías N1=7 con código-nombre 1:1
canal_vals = silver.select("canal").distinct().count()
sev_vals = silver.filter(F.col("nivel_severidad").isNotNull()).select("nivel_severidad").distinct().count()
n1_pares = silver.select("cod_categoria_n1", "categoria_n1").distinct().count()
n1_cod = silver.select("cod_categoria_n1").distinct().count()
control("5b", "SILVER", "Clasificaciones",
        "Dominios cerrados: canal=7, severidad=4 (+NULL), categorías N1=7 con código-nombre 1:1",
        "OK" if (canal_vals == 7 and sev_vals == 4 and n1_pares == n1_cod == 7) else "FALLA",
        f"canal={canal_vals}, severidad={sev_vals}, n1_pares={n1_pares}, n1_cod={n1_cod}")

# 5c. severidad derivada coherente con los cortes EXACTOS aprobados (p50=367.371, p90=14.217.315, p99=152.000.000)
mal_sev = silver.filter(F.col("nivel_severidad").isNotNull() & ~(
    ((F.col("cuantia_bruta") < 367371) & (F.col("nivel_severidad") == "BAJA")) |
    ((F.col("cuantia_bruta") >= 367371) & (F.col("cuantia_bruta") < 14217315) & (F.col("nivel_severidad") == "MEDIA")) |
    ((F.col("cuantia_bruta") >= 14217315) & (F.col("cuantia_bruta") < 152000000) & (F.col("nivel_severidad") == "ALTA")) |
    ((F.col("cuantia_bruta") >= 152000000) & (F.col("nivel_severidad") == "EXTREMA")))).count()
sev_nulos_con_bruta = silver.filter(F.col("nivel_severidad").isNull() & F.col("cuantia_bruta").isNotNull()).count()
control("5c", "SILVER", "Clasificaciones", "nivel_severidad coherente con los cortes exactos p50/p90/p99",
        "OK" if (mal_sev == 0 and sev_nulos_con_bruta == 0) else "FALLA",
        f"mal_clasificados={mal_sev}, con_bruta_sin_severidad={sev_nulos_con_bruta}")

# 5d. procesos/actividades con más de un nombre (excepción documentada, marcada con banderas)
proc_amb = silver.filter(F.col("flg_proceso_ambiguo")).count()
act_amb = silver.filter(F.col("flg_actividad_ambigua")).count()
control("5d", "SILVER", "Clasificaciones",
        "Procesos/actividades multinombre marcados con banderas (regla: no se reetiquetan)",
        "EXCEPCION",
        f"flg_proceso_ambiguo={proc_amb} (códigos 10.2, 6.3, 11.3), flg_actividad_ambigua={act_amb} (6 códigos)")

# 5e. Gold freq_por_severidad == distribución de severidad de Silver
silv_sev = {r["nivel_severidad"]: r["n"] for r in silver.groupBy(
    F.coalesce(F.col("nivel_severidad"), F.lit("SIN_CUANTIA")).alias("nivel_severidad")
).agg(F.count("*").alias("n")).collect()}
gold_sev = {r["nivel_severidad"]: r["n"] for r in g("freq_por_severidad").select(
    "nivel_severidad", F.col("eventos").alias("n")).collect()}
control("5e", "GOLD", "Clasificaciones", "freq_por_severidad == distribución de severidad en Silver",
        "OK" if silv_sev == gold_sev else "FALLA", f"silver={silv_sev}")

OK        5a   tipo_evento P/B y R/A perfectamente alineados | violaciones=0
OK        5b   Dominios cerrados: canal=7, severidad=4 (+NULL), categorías N1=7 con código-nombre 1:1 | canal=7, severidad=4, n1_pares=7, n1_cod=7
OK        5c   nivel_severidad coherente con los cortes exactos p50/p90/p99 | mal_clasificados=0, con_bruta_sin_severidad=0
EXCEPCION 5d   Procesos/actividades multinombre marcados con banderas (regla: no se reetiquetan) | flg_proceso_ambiguo=9406 (códigos 10.2, 6.3, 11.3), flg_actividad_ambigua=7727 (6 códigos)
OK        5e   freq_por_severidad == distribución de severidad en Silver | silver={'BAJA': 13350, 'SIN_CUANTIA': 25188, 'MEDIA': 10680, 'EXTREMA': 268, 'ALTA': 2403}


In [0]:
# ===== CONTROL 6: CONCILIACIÓN DE CANTIDADES ENTRE CAPAS =====
# 6a. Bronze == Silver (sin eliminaciones ni deduplicación: regla aprobada)
nb, ns = bronze.count(), silver.count()
control("6a", "BRONZE-SILVER", "Conciliación", "Filas Bronze == filas Silver (sin eliminación ni deduplicación)",
        "OK" if nb == ns == 51889 else "FALLA", f"bronze={nb}, silver={ns}")

# 6b. Silver == suma de eventos de cada tabla de frecuencia Gold
det6b, ok6b = [], True
for t in ["freq_eventos_periodo", "freq_por_proceso", "freq_por_categoria", "freq_por_causa", "freq_por_severidad"]:
    s = g(t).agg(F.sum("eventos")).collect()[0][0]
    ok6b &= (s == ns)
    det6b.append(f"{t}={s}")
control("6b", "SILVER-GOLD", "Conciliación", "Suma de eventos de las 5 tablas de frecuencia == Silver",
        "OK" if ok6b else "FALLA", "; ".join(det6b))

# 6c. serie mensual sobre calendario completo (transformación justificada: meses sin eventos = 0)
meses_con = silver.select("anio_evento", "mes_evento").distinct().count()
cal_filas = g("freq_eventos_periodo").count()
cal_cero = g("freq_eventos_periodo").filter(F.col("eventos") == 0).count()
control("6c", "SILVER-GOLD", "Conciliación",
        "Calendario mensual completo 142 filas; meses sin eventos en 0 (convención documentada)",
        "OK" if (cal_filas == 142 and meses_con + cal_cero == 142) else "FALLA",
        f"meses_con_eventos={meses_con}, meses_en_cero={cal_cero}, filas_tabla={cal_filas}")

# 6d. modelo_insumo_severidad == A con cuantía (transformación justificada: 21 A sin cuantía excluidos, regla VaR)
a_total = silver.filter(F.col("codigo_perdida") == "A").count()
a_con = silver.filter((F.col("codigo_perdida") == "A") & F.col("cuantia_bruta").isNotNull()).count()
ms_n = g("modelo_insumo_severidad").count()
control("6d", "SILVER-GOLD", "Conciliación",
        "modelo_insumo_severidad == eventos A con cuantía (21 A sin cuantía excluidos por regla VaR)",
        "OK" if ms_n == a_con == 2046 else "FALLA",
        f"A_total={a_total}, A_con_cuantia={a_con}, tabla={ms_n}, excluidos_justificados={a_total - a_con}")

# 6e. panel de frecuencia: 7 categorías x 142 meses, suma de eventos == Silver
pf = g("modelo_insumo_frecuencia").agg(F.count("*").alias("filas"), F.sum("eventos").alias("e")).collect()[0]
control("6e", "GOLD", "Conciliación", "Panel 994 filas (7 categorías x 142 meses) y suma de eventos == Silver",
        "OK" if pf["filas"] == 994 and pf["e"] == ns else "FALLA", f"filas={pf['filas']}, eventos={pf['e']}")

# 6f. perdidas_por_segmento == pares (año, categoría N1) reales en Silver
pares = silver.select("anio_evento", "cod_categoria_n1").distinct().count()
ps_n = g("perdidas_por_segmento").count()
control("6f", "SILVER-GOLD", "Conciliación", "perdidas_por_segmento == pares año-categoría presentes en Silver",
        "OK" if pares == ps_n else "FALLA", f"pares_silver={pares}, tabla={ps_n}")

# ===== CONTROL 7: CONSISTENCIA DE MONTOS AGREGADOS =====
# 7a. suma de cuantía bruta idéntica en Silver y en los 4 agregados Gold
sumas = {"silver": float(silver.agg(F.sum("cuantia_bruta")).collect()[0][0])}
for t in ["perdidas_por_periodo", "perdidas_por_segmento", "series_frecuencia_severidad", "ind_severidad_global"]:
    sumas[t] = float(g(t).agg(F.sum("suma_cuantia_bruta")).collect()[0][0])
ok7a = all(abs(v - 241789756996.00) < 0.01 for v in sumas.values())
control("7a", "SILVER-GOLD", "Montos agregados",
        "Suma de cuantía bruta == 241.789.756.996 COP en Silver y los 4 agregados Gold",
        "OK" if ok7a else "FALLA", "; ".join(f"{k}={v:.2f}" for k, v in sumas.items()))

# 7b. sumas de neta y recuperada consistentes entre Silver, período y segmento
sil7 = silver.agg(F.sum("cuantia_neta").alias("neta"), F.sum("cuantia_total_recuperada").alias("rec")).collect()[0]
p7 = g("perdidas_por_periodo").agg(F.sum("suma_cuantia_neta").alias("neta"), F.sum("suma_total_recuperada").alias("rec")).collect()[0]
s7 = g("perdidas_por_segmento").agg(F.sum("suma_cuantia_neta").alias("neta"), F.sum("suma_total_recuperada").alias("rec")).collect()[0]
ok7b = (float(sil7["neta"]) == float(p7["neta"]) == float(s7["neta"]) and
        float(sil7["rec"]) == float(p7["rec"]) == float(s7["rec"]))
control("7b", "SILVER-GOLD", "Montos agregados", "Sumas de neta y recuperada idénticas en Silver, período y segmento",
        "OK" if ok7b else "FALLA",
        f"netas: silver={sil7['neta']}, periodo={p7['neta']}, segmento={s7['neta']} | recuperada: {sil7['rec']} / {p7['rec']} / {s7['rec']}")

# 7c. insumo VaR: suma de bruta de A == suma en modelo_insumo_severidad
a_sum = float(silver.filter((F.col("codigo_perdida") == "A") & F.col("cuantia_bruta").isNotNull())
                        .agg(F.sum("cuantia_bruta")).collect()[0][0])
ms_sum = float(g("modelo_insumo_severidad").agg(F.sum("cuantia_bruta")).collect()[0][0])
control("7c", "SILVER-GOLD", "Montos agregados", "Suma de bruta de A (insumo VaR) idéntica en Silver y Gold",
        "OK" if a_sum == ms_sum else "FALLA", f"silver={a_sum:.2f}, gold={ms_sum:.2f}")

# 7d. indicadores anuales: suma de eventos con cuantía == Silver
ia = g("ind_severidad_anual").agg(F.sum("eventos_con_cuantia")).collect()[0][0]
con_q = silver.filter(F.col("cuantia_bruta").isNotNull()).count()
control("7d", "SILVER-GOLD", "Montos agregados", "Suma de eventos con cuantía en indicadores anuales == Silver",
        "OK" if ia == con_q == 26701 else "FALLA", f"gold={ia}, silver={con_q}")

OK        6a   Filas Bronze == filas Silver (sin eliminación ni deduplicación) | bronze=51889, silver=51889
OK        6b   Suma de eventos de las 5 tablas de frecuencia == Silver | freq_eventos_periodo=51889; freq_por_proceso=51889; freq_por_categoria=51889; freq_por_causa=51889; freq_por_severidad=51889
OK        6c   Calendario mensual completo 142 filas; meses sin eventos en 0 (convención documentada) | meses_con_eventos=142, meses_en_cero=0, filas_tabla=142
OK        6d   modelo_insumo_severidad == eventos A con cuantía (21 A sin cuantía excluidos por regla VaR) | A_total=2067, A_con_cuantia=2046, tabla=2046, excluidos_justificados=21
OK        6e   Panel 994 filas (7 categorías x 142 meses) y suma de eventos == Silver | filas=994, eventos=51889
OK        6f   perdidas_por_segmento == pares año-categoría presentes en Silver | pares_silver=85, tabla=85
OK        7a   Suma de cuantía bruta == 241.789.756.996 COP en Silver y los 4 agregados Gold | silver=241789756996.00; perdidas_por_

In [0]:
# ===== CONTROL 8 (PARTE 1): SNAPSHOT ANTES DE RE-EJECUTAR EL PIPELINE =====
def metricas():
    """Métricas de negocio (re-lee las tablas en cada llamada para ver la versión vigente)."""
    sil = spark.table(TABLA_SILVER)
    m = {}
    m["silver_filas"] = sil.count()
    m["silver_sum_bruta"] = float(sil.agg(F.sum("cuantia_bruta")).collect()[0][0])
    m["silver_sum_neta"] = float(sil.agg(F.sum("cuantia_neta")).collect()[0][0])
    m["silver_mediana_bruta"] = float(sil.filter(F.col("cuantia_bruta").isNotNull())
                                       .agg(F.percentile("cuantia_bruta", 0.5)).collect()[0][0])
    m["gold_freq_total"] = g("freq_eventos_periodo").agg(F.sum("eventos")).collect()[0][0]
    m["gold_perd_sum_bruta"] = float(g("perdidas_por_periodo").agg(F.sum("suma_cuantia_bruta")).collect()[0][0])
    m["gold_ind_mediana"] = float(g("ind_severidad_global").agg(F.min("mediana")).collect()[0][0])
    m["gold_ms_sum"] = float(g("modelo_insumo_severidad").agg(F.sum("cuantia_bruta")).collect()[0][0])
    m["gold_series_filas"] = g("series_frecuencia_severidad").count()
    return m

ANTES = metricas()
ANTES_LOTE = (spark.table(TABLA_SILVER).select("_lote_silver").limit(1).collect()[0][0],
              g("freq_eventos_periodo").select("_lote_gold").limit(1).collect()[0][0])

(spark.createDataFrame([{"metrica": k, "valor": float(v)} for k, v in ANTES.items()])
   .withColumn("fase", F.lit("ANTES")).withColumn("_lote_qa", F.lit(LOTE_QA))
   .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SCHEMA_GOLD}.controles_snapshot"))
spark.sql(f"COMMENT ON TABLE {SCHEMA_GOLD}.controles_snapshot IS "
          "'Evidencia del control de idempotencia: métricas de negocio antes y después de re-ejecutar el pipeline'")

print("Snapshot ANTES capturado. Lotes actuales:", ANTES_LOTE)
for k, v in ANTES.items():
    print(f"  {k} = {v}")
print("\n>>> RE-EJECUTAR AHORA: notebook 03 (Silver) completo y notebook 04 (Gold) completo, sin cambiar reglas <<<")

Snapshot ANTES capturado. Lotes actuales: ('silver_20261121_203616', 'gold_20261121_203820')
  silver_filas = 51889
  silver_sum_bruta = 241789756996.0
  silver_sum_neta = 240219641258.0
  silver_mediana_bruta = 367371.0
  gold_freq_total = 51889
  gold_perd_sum_bruta = 241789756996.0
  gold_ind_mediana = 367371.0
  gold_ms_sum = 5335174360.0
  gold_series_filas = 142

>>> RE-EJECUTAR AHORA: notebook 03 (Silver) completo y notebook 04 (Gold) completo, sin cambiar reglas <<<


In [0]:
# ===== CONTROL 8 (PARTE 2): VERIFICACIÓN TRAS RE-EJECUTAR EL PIPELINE =====
# (Esta celda se ejecuta DESPUÉS de re-ejecutar los notebooks 03 y 04 sin cambiar ninguna regla)
DESPUES = metricas()
DESPUES_LOTE = (spark.table(TABLA_SILVER).select("_lote_silver").limit(1).collect()[0][0],
                 g("freq_eventos_periodo").select("_lote_gold").limit(1).collect()[0][0])

# 8a. idempotencia: las métricas de negocio son idénticas tras re-ejecutar
dif = {k: (ANTES[k], DESPUES[k]) for k in ANTES if ANTES[k] != DESPUES[k]}
control("8a", "PIPELINE", "Re-ejecución",
        "Re-ejecutar el pipeline produce resultados idénticos (métricas de negocio)",
        "OK" if not dif else "FALLA", f"cambios={dif if dif else 'ninguno'}")

# 8b. trazabilidad: los lotes cambiaron (evidencia de que el re-procesamiento ocurrió)
control("8b", "PIPELINE", "Re-ejecución",
        "Lotes de procesamiento nuevos tras re-ejecutar (trazabilidad del re-procesamiento)",
        "OK" if DESPUES_LOTE != ANTES_LOTE else "FALLA",
        f"silver: {ANTES_LOTE[0]} -> {DESPUES_LOTE[0]} | gold: {ANTES_LOTE[1]} -> {DESPUES_LOTE[1]}")

# Evidencia DESPUÉS en la tabla snapshot
(spark.createDataFrame([{"metrica": k, "valor": float(v)} for k, v in DESPUES.items()])
   .withColumn("fase", F.lit("DESPUES")).withColumn("_lote_qa", F.lit(LOTE_QA))
   .write.mode("append").saveAsTable(f"{SCHEMA_GOLD}.controles_snapshot"))

# ===== ESCRITURA DE LA TABLA DE RESULTADOS DE VALIDACIÓN =====
df_resultados = spark.createDataFrame(RESULTADOS)
(df_resultados.write.mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable(f"{SCHEMA_GOLD}.controles_calidad"))
spark.sql(f"COMMENT ON TABLE {SCHEMA_GOLD}.controles_calidad IS "
          "'Resultados de los controles de calidad del pipeline: OK / EXCEPCION / FALLA con detalle, por control y capa'")

print()
print("=== RESUMEN DE CONTROLES ===")
display(spark.table(f"{SCHEMA_GOLD}.controles_calidad").groupBy("resultado").count())
fallas = spark.table(f"{SCHEMA_GOLD}.controles_calidad").filter(F.col("resultado") == "FALLA").count()
print("Controles FALLA:", fallas, "(0 = pipeline validado y listo para el análisis estadístico)")

OK        8a   Re-ejecutar el pipeline produce resultados idénticos (métricas de negocio) | cambios=ninguno
OK        8b   Lotes de procesamiento nuevos tras re-ejecutar (trazabilidad del re-procesamiento) | silver: silver_20261121_203616 -> silver_20261121_210607 | gold: gold_20261121_203820 -> gold_20261121_210730

=== RESUMEN DE CONTROLES ===


resultado,count
OK,27
EXCEPCION,3
INFO,1


Controles FALLA: 0 (0 = pipeline validado y listo para el análisis estadístico)


## Registro de la etapa de controles (21-Nov-2026)

### Resultado global
**31 controles ejecutados: 27 OK · 3 EXCEPCION (documentadas) · 1 INFO · 0 FALLA.**
Pipeline Bronze → Silver → Gold validado y listo para el análisis estadístico.

### Tablas creadas en esta etapa
| Tabla | Filas | Contenido |
| --- | --- | --- |
| controles_calidad | 31 | Resultado de cada control: id, capa, categoría, resultado (OK/EXCEPCION/INFO/FALLA), detalle, fecha y lote |
| controles_snapshot | 18 | Métricas de negocio ANTES y DESPUÉS de re-ejecutar el pipeline (evidencia del control de idempotencia) |

### Controles ejecutados y hallazgos
1. **Identificadores:** id_evento único 1..51.889 sin nulos; idinsumo con 23.026 nulos y 513 filas repetidas (EXCEPCION documentada: idinsumo no es clave única, por eso Silver genera id_evento); modelo_insumo_severidad con id_evento válido y padre en Silver.
2. **Fechas y montos:** fecha_inicio 100% presente; 0 violaciones de fecha_fin < fecha_inicio en 51.889 filas con ambas fechas; rango 2000-2026 sin valores fuera; sin negativos en bruta/recuperada ni ceros; 28 netas negativas marcadas (flg); neta == bruta - recuperada en las 80 filas con las tres cuantías (0 inconsistencias); moneda PESOS única.
3. **Duplicados:** id_evento sin duplicados; 953 grupos de filas idénticas en columnas de negocio (1.985 filas) conservadas por regla aprobada (EXCEPCION documentada); claves de agregación de las 9 tablas Gold sin duplicados.
4. **Obligatorios:** 7 campos obligatorios del evento 100% presentes; completitud informativa (INFO): cuantia_bruta 48,5% nulos, cuantia_neta 48,8%, fecha_contabilizacion 96,0%, fecha_recuperacion 99,5%, idinsumo 44,4% — ausencias estructurales de la fuente, nunca imputadas.
5. **Clasificaciones:** P/B y R/A perfectamente alineados (0 violaciones); dominios cerrados (canal 7, severidad 4 + NULL, categorías N1 7 con código-nombre 1:1); severidad coherente con los cortes exactos (0 mal clasificados); 9.406 flg_proceso_ambiguo + 7.727 flg_actividad_ambigua (EXCEPCION: multinombres reales, no se reetiquetan); freq_por_severidad == distribución de Silver.
6. **Conciliación de conteos:** Bronze == Silver == 51.889; las 5 tablas de frecuencia Gold suman 51.889; 142 meses de calendario (todos con eventos); A: 2.067 total → 2.046 con cuantía (21 excluidos por regla VaR, transformación justificada); panel 994 filas; perdidas_por_segmento == 85 pares año-categoría reales.
7. **Montos agregados:** Σ bruta 241.789.756.996 COP idéntica en Silver y 4 agregados Gold; Σ neta 240.219.641.258 y Σ recuperada 1.549.331.546 idénticas en Silver, período y segmento; Σ bruta de A (insumo VaR) 5.335.174.360 en ambas capas; indicadores anuales == 26.701 eventos con cuantía.
8. **Re-ejecución (idempotencia):** tras re-ejecutar los notebooks 03 y 04 sin cambiar ninguna regla, las 9 métricas de negocio comparadas son idénticas (cambios=ninguno); los lotes cambiaron (silver_20261121_203616 → silver_20261121_210607; gold_20261121_203820 → gold_20261121_210730): evidencia de trazabilidad del re-procesamiento.

### Nota de reglas
**Ninguna regla de negocio fue modificada durante los controles.** Las EXCEPCIONES corresponden a desviaciones reales de la fuente ya documentadas en los registros de decisiones de Silver y Gold; los controles solo leen y reportan.